In [ ]:
class word2vec():
    def __init__(self, num_negative=5):
        self.n = settings['n']
        self.eta = settings['learning_rate']
        self.epochs = settings['epochs']
        self.window = settings['window_size']
        self.num_negative = num_negative  # Number of negative samples per positive
    
    
    # GENERATE SKIP-GRAM TRAINING DATA (with integer indices, not one-hot)
    def generate_training_data(self, settings, corpus):
        # GENERATE WORD COUNTS
        word_counts = defaultdict(int)
        for row in corpus:
            for word in row:
                word_counts[word] += 1

        self.v_count = len(word_counts.keys())

        # GENERATE LOOKUP DICTIONARIES
        self.words_list = sorted(list(word_counts.keys()), reverse=False)
        self.word_index = dict((word, i) for i, word in enumerate(self.words_list))
        self.index_word = dict((i, word) for i, word in enumerate(self.words_list))
        
        # BUILD NEGATIVE SAMPLING DISTRIBUTION (unigram^0.75)
        self.word_counts = np.array([word_counts[word] for word in self.words_list])
        self.neg_sampling_probs = self.word_counts ** 0.75
        self.neg_sampling_probs = self.neg_sampling_probs / self.neg_sampling_probs.sum()

        training_data = []
        # CYCLE THROUGH EACH SENTENCE IN CORPUS
        for sentence in corpus:
            sent_len = len(sentence)

            # CYCLE THROUGH EACH WORD IN SENTENCE
            for i, word in enumerate(sentence):
                # Get integer indices (not one-hot vectors!)
                center_idx = self.word_index[sentence[i]]

                # CYCLE THROUGH CONTEXT WINDOW
                for j in range(i - self.window, i + self.window + 1):
                    if j != i and j <= sent_len - 1 and j >= 0:
                        context_idx = self.word_index[sentence[j]]
                        # Store as indices: [center_idx, context_idx]
                        training_data.append([center_idx, context_idx])
        return np.array(training_data, dtype=np.int32)


    # SIGMOID ACTIVATION (for negative sampling)
    def sigmoid(self, x):
        # Clip to prevent overflow
        x = np.clip(x, -500, 500)
        return 1.0 / (1.0 + np.exp(-x))


    # SOFTMAX ACTIVATION FUNCTION (kept for reference, not used with negative sampling)
    def softmax(self, x):
        e_x = np.exp(x - np.max(x))
        return e_x / e_x.sum(axis=0)

    
    # SAMPLE NEGATIVE EXAMPLES
    def sample_negatives(self, positive_idx, num_samples):
        """Sample negative examples different from positive context."""
        negatives = []
        while len(negatives) < num_samples:
            neg_idx = np.random.choice(self.v_count, p=self.neg_sampling_probs)
            if neg_idx != positive_idx:  # Don't sample the positive word
                negatives.append(neg_idx)
        return negatives


    # FORWARD PASS WITH NEGATIVE SAMPLING (using embedding lookup)
    def forward_pass_negative_sampling(self, center_idx, pos_context_idx, neg_context_indices):
        """
        Compute forward pass for negative sampling using direct embedding lookup.
        Returns: positive score, negative scores, center embedding
        """
        # EMBEDDING LOOKUP: Get center word embedding directly from W1
        # Instead of: h = W1^T * one_hot(center_idx)
        # We do: h = W1[center_idx, :] (much faster!)
        h = self.w1[center_idx, :]  # Shape: (n,)
        
        # Positive context: score = sigmoid(h · w2[pos])
        pos_score = self.sigmoid(np.dot(h, self.w2[:, pos_context_idx]))
        
        # Negative contexts: scores = sigmoid(-h · w2[neg])
        neg_scores = []
        for neg_idx in neg_context_indices:
            neg_score = self.sigmoid(-np.dot(h, self.w2[:, neg_idx]))
            neg_scores.append(neg_score)
        
        return pos_score, neg_scores, h


    # BACKPROPAGATION WITH NEGATIVE SAMPLING (using index updates)
    def backprop_negative_sampling(self, center_idx, pos_context_idx, neg_context_indices, 
                                   pos_score, neg_scores, h):
        """
        Backprop with negative sampling - only updates relevant embeddings.
        Loss = -log(σ(h·w_pos)) - Σ log(σ(-h·w_neg))
        """
        # Gradient for positive context
        # d_Loss/d_(h·w_pos) = -(1 - σ(h·w_pos)) = -(1 - pos_score)
        pos_grad = -(1.0 - pos_score)
        
        # Update W2 for positive context: w2[:, pos] -= eta * pos_grad * h
        self.w2[:, pos_context_idx] -= self.eta * pos_grad * h
        
        # Accumulate gradient for center embedding from positive
        dL_dh = pos_grad * self.w2[:, pos_context_idx]
        
        # Gradients for negative contexts
        # d_Loss/d_(-h·w_neg) = -(1 - σ(-h·w_neg)) = -(1 - neg_score)
        for i, neg_idx in enumerate(neg_context_indices):
            neg_grad = -(1.0 - neg_scores[i])
            
            # Update W2 for negative context: w2[:, neg] -= eta * neg_grad * (-h)
            self.w2[:, neg_idx] -= self.eta * neg_grad * (-h)
            
            # Accumulate gradient for center embedding from negative
            dL_dh += neg_grad * (-self.w2[:, neg_idx])
        
        # DIRECT INDEX UPDATE: Update only the center word's embedding
        # Instead of: W1 -= eta * outer(one_hot, dL_dh)
        # We do: W1[center_idx, :] -= eta * dL_dh (much faster!)
        self.w1[center_idx, :] -= self.eta * dL_dh


    # TRAIN SKIP-GRAM MODEL WITH NEGATIVE SAMPLING (using indices)
    def train(self, training_data):
        # INITIALIZE WEIGHT MATRICES
        self.w1 = np.random.uniform(-0.8, 0.8, (self.v_count, self.n))     # embedding matrix
        self.w2 = np.random.uniform(-0.8, 0.8, (self.n, self.v_count))     # context matrix
        
        # CYCLE THROUGH EACH EPOCH
        for epoch in range(self.epochs):
            self.loss = 0

            # CYCLE THROUGH EACH TRAINING PAIR
            # Each sample is now (center_idx, context_idx) - just integers!
            for center_idx, context_idx in training_data:
                # Sample negative examples (different from positive context)
                neg_context_indices = self.sample_negatives(context_idx, self.num_negative)
                
                # FORWARD PASS with negative sampling (using embedding lookup)
                pos_score, neg_scores, h = self.forward_pass_negative_sampling(
                    center_idx, context_idx, neg_context_indices
                )
                
                # BACKPROPAGATION with negative sampling (using index updates)
                self.backprop_negative_sampling(
                    center_idx, context_idx, neg_context_indices,
                    pos_score, neg_scores, h
                )

                # CALCULATE LOSS 
                # Loss = -log(σ(pos)) - Σ log(σ(neg))
                self.loss += -np.log(pos_score + 1e-10)  # Add small epsilon to prevent log(0)
                self.loss += -sum([np.log(neg_score + 1e-10) for neg_score in neg_scores])
                
            print(f'EPOCH: {epoch}, LOSS: {self.loss:.4f}')


    # input a word, returns a vector (if available)
    def word_vec(self, word):
        w_index = self.word_index[word]
        v_w = self.w1[w_index]
        return v_w

    
    # input a vector, returns nearest word(s)
    def vec_sim(self, vec, top_n):
        # CYCLE THROUGH VOCAB
        word_sim = {}
        for i in range(self.v_count):
            v_w2 = self.w1[i]
            theta_num = np.dot(vec, v_w2)
            theta_den = np.linalg.norm(vec) * np.linalg.norm(v_w2)
            theta = theta_num / theta_den

            word = self.index_word[i]
            word_sim[word] = theta

        words_sorted = sorted(word_sim.items(), key=lambda x: x[1], reverse=True)

        for word, sim in words_sorted[:top_n]:
            print(word, sim)


    # input word, returns top [n] most similar words
    def word_sim(self, word, top_n):
        w1_index = self.word_index[word]
        v_w1 = self.w1[w1_index]

        # CYCLE THROUGH VOCAB
        word_sim = {}
        for i in range(self.v_count):
            v_w2 = self.w1[i]
            theta_num = np.dot(v_w1, v_w2)
            theta_den = np.linalg.norm(v_w1) * np.linalg.norm(v_w2)
            theta = theta_num / theta_den

            word = self.index_word[i]
            word_sim[word] = theta

        words_sorted = sorted(word_sim.items(), key=lambda x: x[1], reverse=True)

        for word, sim in words_sorted[:top_n]:
            print(word, sim)